# v3.13 — the record: automatic writing with non-inheritable meaning

**Spec only. No code. Written against the v3.12 build; v3.12's result may change it. Seven
DECISION points, none settled.**

## What the previous draft got wrong

My earlier draft gave agents a `mark` action and a writer gene, and then pre-registered the null
that `sym_gain` would go to zero because writing costs a step and pays the writer nothing.

**That draft re-ran two results this project has already closed.** Write-probability collapse is
finding #1. Open semantics — a store whose meaning is unconstrained, so nothing binds — is finding
#2. And the "pre-registered null" I proposed *is v1's result*. A spec whose most likely outcome is
a known outcome is not an experiment; it is a re-derivation.

**The restructure removes writing as a decision.** Writing is automatic, costless and universal, so
there is no public-goods problem and no write-probability to collapse. Meaning is constrained by
construction, so semantics are not open. What is left is the only question that was ever open:
**can an agent use, within its life, a mark whose meaning it cannot have inherited?**

## The mechanism

### Writing is automatic

**Every preparation writes.** When an agent performs preparation `k` on food of type `t` at a cell,
the cell records `(label, sign)` for **type `t`**:

- `label = π(k)` — the *label* of the preparation used, not the preparation index;
- `sign = +1` if the preparation was correct, `−1` if not.

There is **no `mark` action, no writer gene, no writing policy**. Every agent that prepares leaves a
record, whether it wants to or not, at no cost. `mark_pref` from the previous draft is therefore
dropped: there is no writing preference to probe.

### Meaning cannot be inherited

`π` is a **permutation of the K preparations, re-drawn at every remap**, independently of the
mapping itself. So label `j` denotes preparation `π⁻¹(j)`, and *which* preparation that is changes
every era.

This is the whole design. If a mark recorded the preparation index directly, its meaning would be
fixed by the world and a genome could evolve to read it — "channel 3 means prep_3" — and the result
would be inheritance, not transmission. Re-drawing `π` each era makes **the label→preparation
binding unavailable to selection**: no genome can carry it, because it is worthless by the next era.
The population must establish it **within lives**, or not at all.

### Reading is an observation

**K new observation channels**, for the food type underfoot. Channel `j` carries the **age-decayed
sign of the latest mark at label `j`** for that type at that cell. No mark under you, or no food:
zeros.

Reading is not an action. It has no cost and no policy, so a null result cannot be ambiguous between
"cannot read" and "does not bother".

### One heritable gain

**`sym_gain`** — a single heritable scalar multiplying the whole block of K read channels,
**starting near zero**. The population can evolve to attend to the record or to ignore it, exactly
as `nav_dir` / `nav_here` work.

Note what has changed relative to the previous draft: **reading is free and writing is automatic**,
so raising `sym_gain` costs an agent nothing and benefits it directly. There is no altruism in the
loop. If `sym_gain` stays at zero here, it is because the record is *useless*, not because writing
is a public good — and that is a clean negative rather than a re-derivation of finding #1.

> **DECISION 1 — the store's shape.** "The cell for its food type" implies each cell holds a
> `T × K` array of age-decayed signs: one row per food type, one column per label. A cell's food
> type changes over time, and the mark must stay attached to the type it was about, or a mark left
> on type A would be read as evidence about type B. On a 60×60 grid that is 3600 × 3 × 5 = 54,000
> floats — negligible. **Proposal: `T × K` signs per cell.** The alternative — one `K` vector per
> cell with the type implicit — is cheaper and wrong for the reason just given.

> **DECISION 2 — the decay constant.** A mark must **outlive the agent that wrote it**, or nothing
> can pass between agents, and must **not outlive the era**, or it becomes actively misleading when
> `π` and the mapping are redrawn. Those two constraints bracket it. **Proposal: half-life ≈ 1
> lifetime, and ≥ 3 half-lives inside `prep_every` = 700.** This is a pre-check measurement, not a
> guess: report mark age at read time, and the fraction of reads whose latest mark predates the
> current era.

> **DECISION 3 — `sym_gain`'s starting value and scope.** One gain over the whole channel block,
> starting near zero (proposal: same mutation scale as the other scalar genes, initialised at
> 0.05). Should it be allowed to go **negative**? A negative gain is a coherent policy — "do the
> opposite of what the mark says" — and would be the right answer in `noise record`. **Proposal:
> allow negative, and report the sign distribution per arm**; a population that drives it negative
> in the noise arm and positive in the real arm is itself evidence the channel is being read.

## The arms

| arm | store | plasticity | what it isolates |
|---|---|---|---|
| `plastic` | **none** | yes | v3.12's world — the baseline the record must beat |
| `plastic + record` | real | yes | the claim |
| `plastic + noise record` | **labels randomised on write** | yes | the store's mere presence |
| `fixed + record` | real | **no** | whether a genome can use a record without learning |
| **`plastic + record (slow)`** | real, **`label_every` = 3 × `prep_every`** | yes | **whether a label's meaning outliving what it names is what the binding needs** |

`noise record` is the load-bearing control. A store changes the world: channels exist, cells carry
state, decay runs. An arm that improves *because a store exists* is not an arm that improved
*because information passed*. In `noise record` the sign is written at a **random label**, so mark
density, channel statistics and decay are identical and the label→preparation association is
destroyed.

`fixed + record` is the genetic control. With `π` redrawn each era it should get nothing, and if it
does get something, the meaning is leaking through the wiring.

## Gate R — RULED: the matched permutation null

**The record's meaning must not be available to selection.** This is the v3.13 stop row.

**The form I specified was the wrong instrument, and the pre-check showed it.** Pooled MI against
the `noise record` arm fires on both real-record arms (+0.570 and +1.017 bits). That is the
estimator, not a leak:

- **Pooled MI has a floor set by the number of eras.** With E eras a label takes only E meanings,
  so the empirical association cannot wash out however well `π` is doing its job. Measured:
  `plastic + record` pooled **0.677 over ~3 eras and 0.605 over 5** — it decays with era count,
  not toward the noise arm.
- **The noise arm is not a matched comparison.** Its within-era structure differs, so the
  difference mixes "π rotates" with "labels are random within an era".

> **Gate R.** Permute **each era's label axis independently** and pool. Era count, sample sizes and
> within-era structure are all preserved; only cross-era consistency is destroyed — which is
> exactly what the gate asks. **z ≤ 2.0** means the observed pooled association is no stronger than
> chance given the era count: `π` is doing its job and meaning is not inheritable. Above that, the
> run is not read.

Pre-check, 1 seed, 5 eras: `plastic + record` z **+0.44** PASS · `plastic + noise` z **−0.10**
PASS · `fixed + record` z **+2.08** FIRES. The last is the arm where a genome could exploit a
leak, so it is the one to watch; at 1 seed and 5 eras it is not decisive.

Carried over unchanged: **row 0** at `pop < 80` · **row 1a** against v3.1 · **row 1b** as a measured
genetic baseline · **row 1c** standing variation · **rig checks 2(a)–(c)** · founder-free metrics
with founder share printed · each arm's own type-blind level.

## The probes

- **`read_pref(a, t, j, s, k)`** — food type `t` underfoot, channel `j` carrying sign `s`, and
  nothing else: how much does this agent want preparation `k`? The reading side, within-agent, no
  behaviour involved.
- **`store_gain`, learned vs innate, on the current `π`** — how much more the agent prefers the
  preparation the mark **endorses** than the alternatives, evaluated under the era's actual `π`.
  **The learned/innate split is the claim's instrument**, exactly as `prep_gain` innate vs learned
  is now: innate ≈ 0 says the genome cannot read the record (which `π` guarantees), and learned > 0
  says this agent bound it inside its own life.
- **`mark_pref` is dropped.** There is no writing policy.

### The `sym_gain` statistic — RULED

**|gain| in a record arm minus |gain| in the no-record arm, per seed.**

The magnitude, not the sign: the sign is absorbable by `W1`, so it carries no information about
whether the channel is read. And the baseline is not zero — the pre-check found **|gain| rises in
every arm including `plastic`, which has no record at all** (+0.170 plastic, +0.223 record,
**+0.332 noise**). An unused gene grows on drift, and noise grew most. So a rising magnitude on its
own is not evidence of reading, and the no-record arm is the only honest baseline.

**This replaces the tempo follow-up's licensing statistic** — the follow-up is no longer
conditional on anything, it is an arm in this experiment.

**DECISION 5 — RULED.** An agent reads marks it wrote itself. That is memory, not transmission, and
no probe on the standing population separates them. **The newborn line separates them**, because a
newborn has written nothing — every mark it reads was left by someone else. So there are **two
claims, reported separately and NEVER pooled**:

- **binding** — `store_gain` learned vs innate;
- **transmission** — newborn preparations-to-first-correct.

A positive on binding with a null on transmission is a real and reportable outcome: the agent
learned to read a mark, but only its own.

## The claim

**Frozen-population replay**, as in v3.12: era-boundary snapshot, births/deaths/injection disabled,
300 steps, nothing can change but `H`. Three cells, differing only in the store:

| cell | store contents |
|---|---|
| **real** | the marks the population actually left |
| **randomised** | same marks, labels shuffled — presence held, content destroyed |
| **erased** | marks cleared |

> **Claim (binding).** Real exceeds randomised by **≥ 0.10 in every seed**, and the gap is **absent
> in `noise record`**.

Randomised is the primary comparison because it holds the observation statistics fixed; **erased is
reported as the third cell** so the contribution of the channels' mere presence is visible.

> **Claim (transmission) — RULED. Two conditioned lines, replacing the newborn measure.**
>
> **(i) Among FIRST-EVER preparations: P(correct | a positive mark for this type is on the cell)
> vs P(correct | none), per arm.** A first preparation is the agent's genome plus whatever the
> world is telling it — it has learned nothing and written nothing.
>
> **(ii) P(chosen preparation = π⁻¹(strongest positive label) | a positive mark is present),
> against 1/K.** This is *following* the record, measured on behaviour rather than inferred from an
> outcome. An agent can be right for its own reasons; it cannot agree with the mark this often by
> accident.
>
> **Preparations-to-first-correct is kept as CORROBORATING ONLY, over agents that reached 5
> preparations.** Conditioning on reaching 5 is what stops censoring being confounded by short
> lives — which is how the pre-check's version read `fixed + record` as best at 1.586 while its
> population was 261 against 717.

### No self-echo — a property of the design, not a defect

A preparation **consumes the food cell**. So the mark it writes cannot be read for a preparation
until food respawns there, and the reader is then whoever is standing on it. **An agent can never
read its own mark about the food it just prepared.**

This is what makes the self-marking confound structurally weak rather than merely unlikely, and it
is why the two lines above are transmission measures and not memory measures. It is checked in
`record_semantics_selftest` — after a preparation the cell holds no food and the mark is present —
so it is a verified property, not an argument.

**DECISION 6 — RULED.** Preparations 1..n of a life, in a world with the store live against the same
world with **`sym_gain` forced to zero**. Not the store removed: removing it changes the world —
mark density, cell state and decay all go — whereas forcing the gain leaves the world identical and
isolates the *reading*. n = 5, reported as the mean number of preparations to the first correct one
and as the hit on preparation 1 alone.

## Order

1. **v3.12 must read first.** If the learner does not clear its D6 claim in a 60-mapping space,
   there is no within-life competence for a record to carry and v3.13 is premature.
2. **The semantics test extends first**, enumerated as always: the `T × K` store, automatic writing
   on every preparation, `π` redraw at each remap, decay, and the fact that reading is an
   observation and not an action.
3. **Gate R before any outcome row.**

> **DECISION 7 — the pre-registered prediction.** Stated so it can fail: `sym_gain` **rises above
> its starting value in `plastic + record` and not in `noise record`**; `store_gain` **learned > 0
> and innate ≈ 0** in every seed; newborn preparations-to-first-correct **lower with the store
> live**. The honest alternative outcome is that the within-life binding of a label that rotates
> every era is simply harder than the preparation conjunction itself — the agent must learn
> label→preparation *and* preparation→type, from the same signal, inside one era — in which case
> v3.13 nulls on a **capacity** limit, not a public-goods one. That would be a new result, and it
> is the reason this version is worth running where the previous draft was not.

## Slow labels — an ARM, not a follow-up

`π` is redrawn every **3 remaps** rather than every one, so a label's meaning **outlives what it
names** by three eras. This is v3.5's tempo condition: two facts moving at different rates, with
the slower one the thing that has to be learned.

**Meaning is still not inheritable.** A genome fixing on "label *j* means preparation *k*" is right
for three eras and then wrong, far inside evolutionary time — and Gate R's permutation null tests
exactly that, on this arm as on the others.

It is an **arm now, not a follow-up conditional on a null**, so the comparison is made inside one
experiment rather than across two. Nothing else moves with it: gates, probes and claims are
identical across the record arms.

> **The pre-registered reading, recorded before the run.**
>
> | fast (`plastic + record`) | slow (`plastic + record (slow)`) | conclusion |
> |---|---|---|
> | null | **positive** | **capacity limit confirmed, and the tempo condition established.** Binding a label that rotates every era is harder than the conjunction itself; give the label three eras and it binds. |
> | null | null | **the first earned transmission null.** A record handed over free, costless to write, costless to read, its meaning stable for three eras, still carries nothing between agents. |
> | positive | positive | the record works; read the margin for whether tempo helps. |
> | positive | null | would need explaining before anything is claimed — slower meaning should not *hurt*. |

## The world, carried from v3.12 with one change

`T = 3`, `K = 5`, 60 mappings, `prep_every` 700, `prep_value` 1.0 = (K−1)·`prep_fail` 0.25 so the
chance EV of a preparation is exactly zero, `spawn_per_patch` 6.0.

**`max_pop` 800 → 1000.** Selection on `sym_gain` needs fecundity: it is one scalar among several
heritable genes, and a population held at the cap has its reproduction throttled, which is exactly
the pressure that would have to move it. The pre-check found `plastic` at 763 of 800 — 95%, which
would have failed the cap check — so the cap was binding on the arm that is the baseline for the
licensing statistic.

## What is deferred

**Emergent writing is v3.14**, and it is specified only after v3.13 reads. If a population cannot
use a record that is handed to it for free, there is nothing to be gained by asking it to choose to
write one — that ordering is what keeps v3.14 from re-running finding #1 a third time.


## 1. Setup

The sim and the analysis are imported, not inlined. This cell fails loudly if either file is missing.

In [ ]:
# --- Colab check -------------------------------------------------------------
# Upload sim.py and analysis.py next to this notebook (Files pane, or run:
#     from google.colab import files; files.upload()
# and pick both).  Nothing else is needed: pure numpy + matplotlib.
import os, sys

missing = [f for f in ("sim_v3_13.py", "analysis_v3_13.py") if not os.path.exists(f)]
if missing:
    raise SystemExit(f"missing {missing} in {os.getcwd()} -- upload them next to this notebook")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import sim_v3_13 as sim, analysis_v3_13 as A

print("numpy", np.__version__)
print("observation size:", sim.N_IN, " actions:", sim.N_ACTIONS,
      " (eat =", sim.EAT, ", preps =", sim.PREP0, "..", sim.PREP0 + sim.N_PREPS - 1, ")",
      " hidden:", A.WORLD["hidden"], " chance recipe hit:", round(A.CHANCE, 3))
print("\nworld (v3.1 metabolism throughout; the chain from v3.6):")
for k, v in A.WORLD.items():
    print(f"  {k:<20} {v}")
print("\n--- world-semantics self-test (every row of the action x cell table) ---")
if not sim.world_semantics_selftest():
    raise SystemExit("the world does not match the spec; nothing below is meaningful")

print("\n--- D5 readability: preparations per TYPE per era, judged on `fixed` only ---")
print("with T = 3 the food is split three ways, so the criterion (>= 3 per type per era) is the")
print("one most likely to fail.  It failed at spawn_per_patch 3.0 and passes at 6.0.")
print(f"  spawn_per_patch = {A.WORLD['spawn_per_patch']}   type_spawn_w = {A.WORLD.get('type_spawn_w')}")
print("\n--- economics ---")
_K, _pf, _pv = sim.N_PREPS, A.WORLD["prep_fail"], A.WORLD["prep_value"]
print(f"T = {sim.N_TYPES}, K = {_K}, mappings = {len(sim.all_mappings())}, N_ACTIONS = {sim.N_ACTIONS}")
print(f"prep_value {_pv} = (K-1) * prep_fail {(_K-1)*_pf};  chance EV "
      f"{(1/_K)*_pv - ((_K-1)/_K)*_pf:+.4f};  chance {1/_K:.3f}, type-blind {1/sim.N_TYPES:.3f}")

print("\n--- founder-tag self-test ---")
print("injected agents are fresh random genomes; their OWN events are excluded from every")
print("event-weighted metric, their children's are not.  The test forces injection so the")
print("exclusion path is actually exercised -- passing on a run with no injections proves nothing.")
if not sim.founder_tag_selftest():
    raise SystemExit("the founder tag is not wired correctly; every founder-free number is suspect")

print("\n--- replay-mapping self-test ---")
print("a knockout that re-seeds the world does NOT get the mapping its genomes were selected")
print("under.  This checks that force_mapping pins it, and that a re-seeded replay can differ")
print("from the source's final mapping -- the condition that made the old knockout misread.")
if not sim.replay_mapping_selftest():
    raise SystemExit("the replay does not carry the mapping it is given; row 3b is meaningless")

print("\n--- frozen-replay self-test ---")
print("row 3b replays an era-boundary snapshot with births, deaths and injection disabled, so")
print("nothing can change but H.  With learning OFF the hit rate must not move across the")
print("window; if it does, the eta 1 side cannot be read as learning.")
if not A.frozen_selftest():
    raise SystemExit("the frozen replay is not frozen; row 3b would not be an attribution")

print("\n--- learning-rule self-test ---")
print("one agent, one fixed observation, one chosen action; action_noise = 0 so act() is")
print("deterministic and the logit checked belongs to the action that laid the trace.")
if not sim.learning_rule_selftest():
    raise SystemExit("the learning rule is not behaving; nothing below is meaningful")

print("\nconditions:")
for name, spec in A.VARIANTS.items():
    ph = " -> ".join(f"{p['n_steps']} steps chain={p['chain']}" for p in spec["phases"])
    print(f"  {name:<24} {ph}")
    print(f"  {'':<24} {({k: v for k, v in spec['kw'].items() if k not in A.WORLD})}")


## 2. Run — staged

`MODE` picks the stage. All stages share **one checkpoint** and each **skips any (arm, seed) already in it**, so `"grid"` continues from the acceptance checkpoint rather than redoing it.

| stage | arms | seeds | runs | est. wall clock |
|---|---|---|---|---|
| `quick` | 3 core | 0 | 3 | ~5 min |
| `acceptance` | fixed, scrambled, plastic (W2) | 0–1 | 6 | ~60–75 min |
| `grid` | + random policy, ceiling | 0–2 | 9 more | **~75–90 min** |

### Read the checkpoint audit before you read anything else

The cell prints an audit of the checkpoint it loads. Runs written **before** a field existed cannot be re-analysed for the reads that depend on it — those counters are accumulated inside the sim, not derived from the log — and `final_mapping` is not reconstructable offline at all, because `World` shares its rng with the agents, so the mapping draw sequence depends on every action-noise draw in the run.

The affected reads are **row 3b (the attribution line)**, **first-preparation hit late-in-era**, **the survivor-conditioned since-remap curve**, and **the survivor halves 1–2 vs 6–10** (older runs recorded the 1–5 split under the same name). They print `nan` rather than a wrong number.

To recover them, put the affected pairs in `REFRESH` — for the v3.11 acceptance checkpoint that is the three core arms at seeds 0–1, six runs, roughly an extra 60–75 min. Everything else (founder-free hit rates, rig check 2(a) on whole-phase safe rate, the abstention rule, the per-era A+B sums, `prep_gain innate`) reads correctly off the existing checkpoint without a refresh.

The cell checkpoints after **every run**, so a dropped session costs one run; just re-run it to resume. **Seeds 3–4 are held in reserve** — set `SEEDS = [3, 4]` after the grid and re-run; the seed criterion adapts (`min(4, n_seeds)`).

In [ ]:
# MODE picks the stage.  All stages share ONE checkpoint and each SKIPS any (arm, seed)
# already in it, so "grid" continues from the acceptance checkpoint rather than redoing it.
#
#   "quick"       smoke test, 3 arms, 1 seed, 1500-step phases.            ~5 min
#   "acceptance"  fixed / scrambled / plastic (W2) x seeds 0-1, full.      ~60-75 min
#   "grid"        continues: adds `random policy` and `fixed + B (ceiling)`
#                 for seeds 0-1, and all five arms for seed 2.  Nine runs.  ~75-90 min
MODE = "acceptance"

CKPT = "results_v3_13.pkl"
CORE = ["plastic", "plastic + record", "plastic + record (slow)"]
ALL  = list(A.VARIANTS)
REFRESH = []

if MODE == "quick":
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0], 1500, CORE, dict(prep_every=300, recipe_every=500)
    CKPT = "results_v3_13_quick.pkl"
elif MODE == "acceptance":
    # The three arms the claim rests on: the no-record baseline, the record, and the record with
    # a label meaning that outlives what it names.  `plastic + noise` and `fixed + record` are the
    # controls and join at the grid stage.
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0, 1], 8000, CORE, {}
elif MODE == "grid":
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0, 1, 2], 8000, ALL, {}
else:
    raise SystemExit(f"MODE must be quick / acceptance / grid, not {MODE!r}")

variants = {k: A.VARIANTS[k] for k in ARMS}
existing = A.load(CKPT)
if existing:
    A.checkpoint_audit(existing)
    print()

variants = {k: A.VARIANTS[k] for k in ARMS}
results = A.run_experiment(seeds=SEEDS, phase_steps=PHASE_STEPS, variants=variants,
                           results=existing, save_path=CKPT, refresh=REFRESH, **OVERRIDES)
print("\nin the checkpoint:", {k: sorted(r["cfg"]["seed"] for r in v) for k, v in results.items()})


## 3. The printed summary

Phase 1 and phase 2 blocks separately, then per-seed values, then the decision numbers in the order of the table above.

**Read row 1 first — it is a stop condition.** If phase 1 does not reproduce v3.1, nothing in phase 2 is readable.

**Reading a QUICK run:** a smoke test, not a result. Phases of 1500 steps are a handful of generations, so row 0 will flag conditions and row 1 will not reproduce. Read it only to confirm every cell produces the output it should.

In [ ]:
A.checkpoint_audit(results)
A.v313_precheck(results)


## 4. The transition

500-step bins across 2000 steps either side of the chain switching on. This is where a collapse becomes visible, and where `eta2`/`lam2` either hold or start falling as they did in v3.6.

In [ ]:
A.transition_table(results, bin_size=500, span=2000)


## 5. Curves

`meal` is v3.1's within-life food curve, read on **phase 1** — the positive control. `att` and `rec` are the recipe curves, read on **phase 2**; row 4 requires `att` to rise (or `hit_old` > `hit_young`).

In [ ]:
A.curves(results, "meal", "meal number in an agent's life")        # phase 1
A.curves(results, "att",  "attempt number in an agent's life")     # phase 2
A.curves(results, "rec",  "attempts since the last recipe change")  # phase 2


## 6. Plots

Solid black line = the chain switches on; dashed = recipe change.

In [ ]:
A.plot_results(results)


## Reading v3.13

**The question:** can an agent use, within its life, a mark whose meaning it cannot have inherited?

Writing is **automatic, costless and universal** — every preparation writes `(label, sign)` for its
food type, with no mark action and no writer gene, so there is no public good and no
write-probability to collapse. Labels are a permutation `π` **redrawn at every remap**, so no genome
can carry what a label means. Reading is an **observation**, so a null cannot be ambiguous between
"cannot read" and "does not bother".

### Read in this order

1. **Gate R** — the matched permutation null, grouped by **π-epoch**. `z ≤ 2.0` means the observed
   cross-era association is no stronger than chance given the epoch count: `π` is doing its job.
   Note the grouping is by epoch, not era, or the slow arm would fire on its own definition.
2. **`sym_gain`** — |gain| in a record arm **minus the no-record arm's**. |gain| rises everywhere on
   drift, so the no-record arm is the baseline.
3. **Transmission (i)** — first-ever preparations split by whether a positive mark was present.
4. **Transmission (ii)** — following, split by whether the mark endorses the correct preparation,
   with the **stale** cell read against `(1 − hit)/(K − 1)`.
5. Preparations-to-first-correct, corroborating, over agents that reached 5.

### What the pre-check established about the instruments

Two of the specified lines were confounded, and the controls caught both.

**Line (i) is confounded and should be read with that in mind.** A positive mark exists only where
someone recently *succeeded*, so it marks places and times where success is common. The gaps are
positive in every record arm — and **largest in `noise`** (+0.241 against `record`'s +0.120), which
is the proof: the noise arm's labels carry nothing, so the gap cannot be about mark content.

**Line (ii) needed two corrections.** Splitting by whether the mark endorses the correct
preparation separates "read the mark" from "was simply right". And `1/K` is the wrong null for the
stale cell: an agent that knows the answer never agrees with a stale mark, whatever it reads. The
null is `(1 − hit)/(K − 1)`.

Read that way, the pre-check (1 seed, 3000-step phases — **not a result**) shows:

| arm | stale follow | null | ratio |
|---|---|---|---|
| `plastic + record` | 0.041 | 0.124 | **0.33** |
| `plastic + noise` | 0.096 | 0.123 | 0.78 |
| `fixed + record` | 0.051 | 0.147 | 0.35 |
| `plastic + record (slow)` | 0.353 | 0.122 | **2.89** |

Ratio above 1 means the agent **follows** a mark it should not; below 1 means it **avoids** it.
Either way the label was read — you cannot avoid what you cannot see. The noise arm at 0.78 is the
reference for how far from 1 an unread channel sits.
